# Complex Numbers, Wirtinger Derivatives, and Signals

This notebook supports the Part 1 article. It focuses on visuals while calling reusable Python modules for the core math and data generation.

In [1]:
import numpy as np
from IPython.display import Markdown, display

from complex_core import CONSTELLATIONS, from_dxdy, conj_from_dxdy
from complex_plots import (
    complex_activation_notes_markdown,
    format_holomorphic_check_markdown,
    format_wirtinger_update_table_markdown,
    format_wirtinger_steps_markdown,
    map_degrees_of_freedom_markdown,
    modulation_notes_markdown,
    plot_constellation_rotation_grid,
    plot_descent_trajectory_comparison,
    plot_iq_impairments,
    plot_iq_map_geometry,
    plot_iq_sample_efficiency,
    plot_rotation_vectors,
    plot_sinusoid_and_iq,
    plot_wirtinger_update_directions,
)

## 1) Imaginary numbers as rotation

Multiplying by `i` rotates a point by 90 degrees. Multiplying by `e^(i*theta)` rotates by any angle.

In [2]:
fig = plot_rotation_vectors(x=1.0, y=1.0, angle_deg=45.0)
fig.show()

## 2) Real derivatives vs Wirtinger derivatives

For `z = x + iy`, Wirtinger operators are:

- `d/dz  = 1/2 (d/dx - i d/dy)`
- `d/dz* = 1/2 (d/dx + i d/dy)`

### Notation and terms cheat sheet

| Term | Meaning in this notebook |
|---|---|
| `z`, `w` | Complex variable/parameter (`z = x + iy`, `w = u + iv`). |
| `x, y` (or `u, v`) | Real and imaginary coordinates of a complex value. |
| `Re(z)`, `Im(z)` | Real part and imaginary part of `z`. |
| `z*` or `conj(z)` | Complex conjugate (`x - iy`). |
| `L` | Real-valued loss function we minimize. |
| `dL/dx`, `dL/dy` | Ordinary real partial derivatives of the loss. |
| `dL/dz` | Wirtinger derivative wrt `z`: `0.5*(dL/dx - i dL/dy)`. |
| `dL/dz*` | Conjugate Wirtinger derivative: `0.5*(dL/dx + i dL/dy)`. For real losses, this gives the complex update direction. |
| `w <- w - lr * dL/dw*` | Complex gradient-descent step used in the notebook. |
| split real/imag update | `u <- u - (lr/2)dL/du`, `v <- v - (lr/2)dL/dv`; equivalent to the `dL/dw*` update here. |
| `lr` | Learning rate (step size). |
| `|z|`, `|z|^2` | Magnitude and squared magnitude (`|z|^2 = z z*`). |
| holomorphic | Function that satisfies Cauchy-Riemann conditions; ordinary complex derivative is valid. |
| Cauchy-Riemann residual | Numeric check for how closely a function behaves holomorphically at a point. |
| IQ | In-phase (`I`) and quadrature (`Q`) components (real/imag axes of a complex signal). |
| global phase rotation `R(x)` | Multiply all samples by `exp(i*theta)`; rotates IQ points without changing class identity. |
| commutation check `f(Rx)` vs `R(fx)` | Tests whether a map preserves phase-rotation structure. |
| MSE | Mean squared error objective used in toy regressions. |


In [3]:
x, y = 1.2, -0.7
df_dx, df_dy = 2 * x, 2 * y  # for f(z)=|z|^2
print('Real partials:')
print('  df/dx =', df_dx)
print('  df/dy =', df_dy)
print('Wirtinger:')
print('  df/dz  =', from_dxdy(df_dx, df_dy))
print('  df/dz* =', conj_from_dxdy(df_dx, df_dy))

Real partials:
  df/dx = 2.4
  df/dy = -1.4
Wirtinger:
  df/dz  = (1.2+0.7j)
  df/dz* = (1.2-0.7j)


In [4]:
display(Markdown(format_wirtinger_steps_markdown('abs2', x=1.2, y=-0.7)))

**Given:** `z = 1.200 + (-0.700)i = 1.200-0.700j`

**Function:** `f(z) = |z|^2 = x^2 + y^2`

**Step 1:** `f(z) = 1.9300`

**Step 2:** `df/dx = 2.4000`, `df/dy = -1.4000`

**Step 3:**
`df/dz  = 0.5*(df/dx - i*df/dy) = 1.2000+0.7000j`

`df/dz* = 0.5*(df/dx + i*df/dy) = 1.2000-0.7000j`


## 2.5) Why ordinary complex derivatives are not enough

Ordinary complex derivatives only work for holomorphic functions. That is a strong requirement: the function must locally behave like rotation+scale and satisfy the Cauchy-Riemann equations.

Most ML losses are real-valued, like `|prediction-target|^2`. These depend on both `z` and `z*`, so ordinary `d/dz` calculus is not enough. Wirtinger calculus keeps `z` and `z*` as separate coordinates.

In [5]:
display(Markdown(format_holomorphic_check_markdown(1.2 - 0.7j)))

Cauchy-Riemann residuals at `z=1.200-0.700j`:

| Function | residual | Interpretation |
|---|---:|---|
| `f(z)=z^2` | `2.00e-05` | holomorphic: ordinary complex derivative works |
| `f(z)=z*` | `2.00e+00` | anti-holomorphic: depends on `z*` |
| `f(z)=|z|^2` | `3.80e+00` | real loss shape: depends on both `z` and `z*` |

Small residual means the ordinary complex derivative is valid. ML losses like `|z|^2` are real-valued and non-holomorphic, so Wirtinger calculus is the useful tool.

## 3) Why Wirtinger matters for IQ backprop

Backprop with a complex parameter is still gradient descent on two real coordinates. If `w = u + iv`, the ordinary real gradient is `(dL/du, dL/dv)`. Wirtinger calculus packages those two numbers into complex coordinates:

- `dL/dw* = 0.5 * (dL/du + i dL/dv)` points in the same complex-plane direction as the real gradient.
- `dL/dw = 0.5 * (dL/du - i dL/dv)` is the conjugated direction. For real losses, it is not the optimizer update direction.

With the math definition above, the equivalent split-real update uses `lr/2` on the raw real partials. Many frameworks absorb that constant into their gradient convention or learning rate; the direction is the important part.

This section builds the idea in three steps: a one-step arrow picture, a repeated-step trajectory, then the IQ-geometry reason complex structure can help.

Framework note from current docs:
- PyTorch complex autograd returns the conjugate Wirtinger gradient (`dL/dz*`) for real losses.
- TensorFlow uses the same convention.
- JAX uses a different convention for some complex differentiation APIs; for complex-parameter optimization with real losses you typically conjugate its gradient before updating.

### 3a) One step: complex update is just the split-real update

Use the toy loss `L(w)=|w-a|^2`. Think of `w` as an IQ point: `I=Re(w)` and `Q=Im(w)`. Ordinary gradient descent says to move `I` using `dL/du` and move `Q` using `dL/dv`.

The left panel shows those two split-real moves and the single complex step `-dL/dw*`. They land at the same point. The right panel shows why `-dL/dw` is wrong: it conjugates the gradient and sends the Q update in the opposite direction.

In [6]:
display(Markdown(format_wirtinger_update_table_markdown()))
fig, summary, stats = plot_wirtinger_update_directions()
fig.show()
print(summary)

For `L(w)=|w-a|^2`, use `w=1.000+1.500j`, target `a=1.000+2.000j`, and `lr=0.25`.

| Quantity | Value | Meaning |
|---|---:|---|
| `w-a` | `0.000-0.500j` | current error vector |
| `dL/du` | `0.000` | real-axis slope |
| `dL/dv` | `-1.000` | imaginary-axis slope |
| `dL/dw*` | `0.000-0.500j` | complex form of the real gradient, scaled by 1/2 |
| `dL/dw` | `0.000+0.500j` | conjugated direction; not the descent update for real losses |

One gradient step:

| Update | New `w` | Loss after step |
|---|---:|---:|
| `w - lr*dL/dw*` | `1.000+1.625j` | `0.1406` |
| split real/imag | `1.000+1.625j` | `0.1406` |
| `w - lr*dL/dw` | `1.000+1.375j` | `0.3906` |

The first two rows match because `dL/dw* = 0.5*dL/du + 0.5i*dL/dv`. The last row flips the imaginary part of the direction, so it moves the parameter away from the target vertically in this example.

Current loss=0.2500. Correct step loss=0.1406; wrong dL/dw step loss=0.3906. The left panel shows that split-real and -dL/dw* land at the same point.


### 3b) Repeated steps: correct update vs wrong conjugated update

Now run multiple steps from a more general starting point. The blue Wirtinger-conjugate path and the green split-real path should sit on top of each other. The red path uses `dL/dw`; it initially may look plausible, but it flips the imaginary part of the update and eventually runs away.

In [7]:
fig, summary, stats = plot_descent_trajectory_comparison(target=1.0 + 2.0j, w_init=-2.0 + 1.5j, lr=0.25, epochs=20)
fig.show()
print(summary)

Final losses -- Wirtinger: 9.302e-05, Split-real: 9.302e-05, Wrong dL/dz: 1.881e+03


### Why split-real works and why `dL/dw` fails

Use the toy loss `L(w)=|w-a|^2` with `w=u+iv` and `a=alpha+i*beta`.

- Real partials are `dL/du = 2(u-alpha)` and `dL/dv = 2(v-beta)`.
- With the math convention used here, split-real gradient descent updates
  `u <- u - (lr/2) dL/du`, `v <- v - (lr/2) dL/dv`,
  so `w <- w - lr[(u-alpha) + i(v-beta)] = w - lr (w-a)`.
- But `(w-a)` is exactly `dL/dw*`, so split-real and Wirtinger-conjugate are the same update.

Why the wrong red update fails:

- `dL/dw = conj(w-a)`, so the wrong update is `w <- w - lr * conj(w-a)`.
- If error `e=w-a=x+iy`, then `e_next = (1-lr)x + i(1+lr)y`.
- Real error shrinks, but imaginary error grows, so loss eventually increases.

### 3c) IQ geometry check: what structure does a complex map add?

This section tests one question: **if we rotate all IQ samples first, then map them, do we get the same result as mapping first, then rotating?**

For a map `f`, we compare `f(Rx)` with `R(fx)` where `R` is a global phase rotation.

How to read the figure:

- Panel 1 shows the original burst `x` and its rotated copy `R(x)`.
- Panel 2 (complex map) overlays `w(Rx)` and `R(wx)`. They should overlap almost perfectly.
- Panel 3 (unconstrained real 2x2 map) shows `A(Rx)` vs `R(Ax)`. They usually separate.

Takeaway: complex multiplication naturally respects IQ global phase rotation, while a generic real 2x2 map does not unless constrained to complex form.

In [8]:
fig, summary, stats = plot_iq_map_geometry(seed=4, n_symbols=256, phase_deg=30.0, delta_deg=45.0)
fig.show()
print(summary)

Commutation error mean ||f(Rx)-R(fx)||^2: complex=1.60e-15, unconstrained real 2x2=7.62e-02, constrained-real(complex form)=1.40e-15. Read the plot left-to-right: panel 2 overlaps (good), panel 3 separates (not rotation-equivariant).


In [9]:
display(Markdown(map_degrees_of_freedom_markdown()))

### Degrees of freedom: why the complex map is constrained

A complex scalar multiply `y = w*x` has 2 real degrees of freedom: `Re(w)` and `Im(w)`. As a real matrix it is always:

```text
[[ Re(w), -Im(w)],
 [ Im(w),  Re(w)]]
```

For `w=0.60+0.90j`, that matrix is approximately:

```text
[[ 0.60, -0.90],
 [ 0.90,  0.60]]
```

A generic real 2x2 map has 4 real degrees of freedom. That extra freedom can learn useful patterns, but it can also shear/warp IQ geometry unless the data teaches it not to.

In [10]:
display(Markdown(complex_activation_notes_markdown()))

### Complex activations caveat

Linear complex layers are straightforward, but nonlinearities need care. A usual real activation like ReLU assumes an ordered real line; complex numbers do not have a natural `positive` direction.

Common complex-network choices include applying nonlinearities to magnitude, phase, or real/imag parts separately. Those choices affect which signal symmetries the model preserves.

Part 2 handles this in the model design; Part 1 only needs the backprop rule and the IQ geometry intuition.

### 3d) When complex structure helps: phase nuisance with limited data

Both formulations can optimize the same real objective.
The difference below is inductive bias: a complex-structured map (rotation+scale form) versus an unconstrained real 2x2 map.
This demonstrates when structure helps, not that real calculus is incorrect.

In [11]:
fig, summary, stats = plot_iq_sample_efficiency(n_train=24, train_phase_deg=15.0, n_test=1024, noise=0.15, seeds=tuple(range(12)), lr=0.3, epochs=35)
fig.show()
print(summary)

Train MSE -- complex: 0.0397, real: 0.0388. Mean test error gap (real - complex): 0.0011. This gap comes from useful structure constraints, not a different objective (complex: 2 real dof, real 2x2: 4 real dof).


## 4) Complex numbers in signals

A complex sinusoid bundles amplitude and phase naturally. A phase shift rotates the whole signal in the IQ plane.

In [12]:
fig = plot_sinusoid_and_iq(amplitude=1.0, frequency=2.0, phase_deg=0.0, snr_db=30.0, shift_deg=45.0, seed=0)
fig.show()

## 4.5) IQ impairments that models need to handle

Real receivers see more than clean constellations. Phase rotation, amplitude scaling, noise, and frequency offset all change the IQ picture without necessarily changing the underlying signal family.

This is the bridge to Part 2: models need either enough data augmentation or enough built-in structure to handle these variations.

In [13]:
fig = plot_iq_impairments(modulation='qpsk', n_symbols=256, seed=3)
fig.show()

## 5) Data preview for Part 2

First we show all classes with no rotation to establish the baseline geometry. Then we apply global rotations and show that the class identity stays the same while the constellation orientation changes.

In [14]:
print('Available constellations:', list(CONSTELLATIONS))
fig = plot_constellation_rotation_grid(rotations_deg=(0.0,), snr_db=20.0, n_symbols=256, seed=0)
fig.show()
display(Markdown(modulation_notes_markdown()))

Available constellations: ['bpsk', 'qpsk', '8psk', '16qam']


### Signal classes used in Part 2

- **BPSK**: Binary phase-shift keying with 2 phase states (1 bit/symbol). Robust low-rate links like telemetry, satellite control, and deep-space style channels.
- **QPSK**: Quadrature phase-shift keying with 4 phase states (2 bits/symbol). Widely used in cellular, satellite, and many digital radio systems.
- **8PSK**: Phase-shift keying with 8 phase states (3 bits/symbol). Used when higher spectral efficiency is needed, for example in some satellite broadcast links.
- **16QAM**: Quadrature amplitude modulation with 16 amplitude-phase points (4 bits/symbol). Common in Wi-Fi, LTE/5G, cable modems, and other high-throughput links.

In [15]:
fig = plot_constellation_rotation_grid(rotations_deg=(0.0, 45.0, 90.0), snr_db=20.0, n_symbols=256, seed=11)
fig.show()

## Wrap-up and Part 2 bridge

Part 1 established the mechanics: complex multiplication encodes rotation+scale, real losses over complex values need the conjugate Wirtinger gradient, and split-real updates are equivalent when implemented correctly.

The practical modeling question is not whether real-valued networks are invalid. It is whether a complex representation gives the right inductive bias for IQ data. Part 2 uses these constellation families to compare real and complex neural-network models under phase rotation and limited data.